In [96]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import xarray as xr
import owncloud
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

### Utility Functions

In [97]:
def train(model: nn.Module, features_train: torch.Tensor, labels_train: torch.Tensor, optimizer, reg_term = 0.0, loss_function = nn.CrossEntropyLoss()) -> None:

    optimizer.zero_grad()

    output = model.forward(features_train)

    loss = loss_function(output, labels_train) + reg_term

    loss.backward()

    optimizer.step()

def calculate_accuracy(model: nn.Module, features: torch.Tensor, labels: torch.Tensor) -> torch.Tensor: 

    output = model(features)

    accuracy = (labels == torch.argmax(output, dim=1)).sum()/len(labels)

    return (accuracy*100).item()

class utils:
    train = train
    calculate_accuracy = calculate_accuracy


### Download Data

In [5]:
Path('data').mkdir(exist_ok=True, parents=True)

owncloud.Client.from_public_link('https://uni-bonn.sciebo.de/s/3Uf2gScrvuTPQhB').get_file('/', f'data/steinmetz_2017-01-08_Muller.nc')

True

Extract data needed to train a classifier that can identify stimulus type (left contrast, right contrast, equal contrast) from spikes

In [6]:
dset = xr.load_dataset('data/steinmetz_2017-01-08_Muller.nc')
dset

<xarray.Dataset> Size: 124MB
Dimensions:             (trial: 261, time: 250, cell: 1268, sample: 82,
                         waveform_component: 3, probe: 384, brain_area_lfp: 5,
                         spike_id: 1836009)
Coordinates:
  * trial               (trial) int32 1kB 1 2 3 4 5 6 ... 257 258 259 260 261
  * time                (time) float64 2kB 0.01 0.02 0.03 0.04 ... 2.48 2.49 2.5
  * cell                (cell) int32 5kB 1 2 3 4 5 ... 1264 1265 1266 1267 1268
  * waveform_component  (waveform_component) int32 12B 1 2 3
  * probe               (probe) int32 2kB 1 2 3 4 5 6 ... 380 381 382 383 384
  * brain_area_lfp      (brain_area_lfp) <U5 100B 'CA1' 'DG' 'LP' 'PO' 'VISam'
  * spike_id            (spike_id) int32 7MB 1 2 3 4 ... 1836007 1836008 1836009
Dimensions without coordinates: sample
Data variables: (12/31)
    contrast_left       (trial) int8 261B 50 0 100 0 50 0 ... 100 0 100 0 100 0
    contrast_right      (trial) int8 261B 0 50 25 100 50 50 ... 100 50 100 25 25
    gocue               (trial) float64 2kB 0.9828 0.902 1.114 ... nan nan nan
    stim_onset          (trial) float64 2kB 0.5 0.5 0.5 0.5 ... 0.5 0.5 0.5 0.5
    feedback_type       (trial) float64 2kB 1.0 1.0 1.0 1.0 ... nan nan nan nan
    feedback_time       (trial) float64 2kB 1.272 1.104 1.402 ... nan nan nan
    ...                  ...
    waveform_w          (cell, sample, waveform_component) float32 1MB 0.0 .....
    waveform_u          (cell, waveform_component, probe) float32 6MB 0.0 ......
    lfp                 (brain_area_lfp, trial, time) float64 3MB -27.6 ... 0...
    spike_time          (spike_id) float32 7MB 2.363 2.385 ... 1.651 0.5142
    spike_cell          (spike_id) uint32 7MB 1 1 1 1 1 ... 1268 1268 1268 1268
    spike_trial         (spike_id) uint32 7MB 1 1 2 2 2 ... 205 205 205 213 252
Attributes:
    session_date:  2017-01-08
    mouse:         Muller
    stim_onset:    0.5
    bin_size:      0.01

In [7]:
spike_cols = ['spike_time', 'spike_cell', 'spike_trial']
df_spikes = dset[spike_cols].to_dataframe().reset_index()

trial_ids = np.sort(df_spikes['spike_trial'].unique())
n_cells = dset.sizes['cell']

# features: spike counts per cell for each trial
features_trials = []
# labels: which side had higher contrast (0=left, 1=right, 2=equal)
labels_trials = []

for trial_id in trial_ids:
    # count spikes per cell
    counts = df_spikes[df_spikes['spike_trial'] == trial_id].groupby('spike_cell').size()
    feature_vector = np.zeros(n_cells)
    feature_vector[counts.index - 1] = counts.values
    features_trials.append(feature_vector)
    
    # create labels
    left = dset['contrast_left'].values[trial_id - 1]
    right = dset['contrast_right'].values[trial_id - 1]
    labels_trials.append(0 if left > right else 1 if right > left else 2)

features = torch.tensor(features_trials, dtype=torch.float32)
labels = torch.tensor(labels_trials, dtype=torch.long)


/tmp/ipykernel_387619/4159512993.py:24: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1752540242079/work/torch/csrc/utils/tensor_new.cpp:254.)
  features = torch.tensor(features_trials, dtype=torch.float32)


In [28]:
SEED = 2026
generator = torch.Generator().manual_seed(SEED)

train_fraction = 0.7
test_fraction = 1-train_fraction

features_train, features_test = random_split(features, lengths=[train_fraction, test_fraction], generator=generator)
labels_train, labels_test = random_split(labels, lengths=[train_fraction, test_fraction], generator=generator)

features_train, features_test = features_train.dataset.float(), features_test.dataset.float()
labels_train, labels_test = labels_train.dataset, labels_test.dataset

## Section 1: Regularize Model


| Code | Description |
|---|---|
| `for param in model.parameters():` | Loop through all parameters in `model`. |
| `l1 = 0.0`<br>`for param in model.parameters():`<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`l1 += torch.sum(torch.abs(param))` | Calculate L1 norm for `model`. |
| `l2 = 0.0`<br>`for param in model.parameters():`<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`l2 += torch.sum(param**2))` | Calculate L2 norm for `model`. |
| `loss = loss_function(prediction, labels) + lam*l1` | L1 regularization: Add L1 penalty to loss function. |
| `loss = loss_function(prediction, labels) + lam*l2` | L2 regularization: Add L2 penalty to loss function. |
| `nn.Dropout(0.1)` | Randomly removes 10 % of the neurons from layer. |

Note: Dropout usually with 50 %.
- Adds noise
- Prevents overfitting by breaking brittle predictions
- Enforces distributed representation - that all nodes are used.
- Forces dead branches to learn - more generalizable, stable predictions


In [440]:
n_classes = 3
# define model with two fully connected layers and one ReLU layer between them
class Model(nn.Module):
    def __init__(self, hidden_layer_sizes = [64,32]):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(n_cells, hidden_layer_sizes[0]),
            nn.ReLU(),
            nn.Linear(hidden_layer_sizes[0], n_classes),
        )

    def forward(self, x):
        return self.layers(x)

In [441]:
# create model
torch.manual_seed(SEED)
model = Model()

params_square_sum = 0.0
for param in model.parameters():
    params_square_sum += torch.sum(param**2)

params_norm = torch.sqrt(params_square_sum)
params_norm

tensor(4.7219, grad_fn=<SqrtBackward0>)

In [ ]:
nepochs = 50

torch.manual_seed(SEED)
model = Model()

optimizer = torch.optim.RMSprop(model.parameters(), lr = 0.01)

lam = 0.1
#lam = 0.0

for epoch in range(nepochs):
    l1 = 0.0
    for param in model.parameters():
        # param in one loop corresponds to all weights of each layer
        l1 += torch.sum(torch.abs(param))

    l2 = 0.0
    for param in model.parameters():
        # param in one loop corresponds to all weights of each layer
        l2 += torch.sum(param**2)

    utils.train(model, features_train, labels_train, reg_term = lam*l2, optimizer=optimizer, )

In [477]:
utils.calculate_accuracy(model, features_test, labels_test)

55.938697814941406

In [478]:
params_square_sum = 0.0
for param in model.parameters():
    params_square_sum += torch.sum(param**2)

params_norm = torch.sqrt(params_square_sum)
params_norm

tensor(26.8904, grad_fn=<SqrtBackward0>)

In [320]:
# train model
utils.train(model, features_train, labels_train)
utils.calculate_accuracy(model, features_test, labels_test)

TypeError: train() missing 1 required positional argument: 'optimizer'

In [145]:
params_square_sum = 0.0
for param in model.parameters():
    params_square_sum += torch.sum(param**2)

params_norm = torch.sqrt(params_square_sum)
params_norm

tensor(5.7898, grad_fn=<SqrtBackward0>)

L1 Norm
- Drives some weights to 0.
- Use when some features are likely to be unimportant

In [ ]:
model = Model()

In [12]:
model = Model()
l1 = 0.0
for param in model.parameters():
  # param in one loop corresponds to all weights of each layer
  print(len(param))
  l1 += torch.sum(torch.abs(param))
l1

128
128
128
128
3
3


tensor(3025.2070, grad_fn=<AddBackward0>)

L2 Norm
- Make the biggest weights smaller.
- Use when you have correlated features
  - W/correlated features the weights for one might be kept and dominate the model while the weights for the other feature go to zero
    - Not good if correlation is not close to 1 - lose info in second feature

In [ ]:
model = Model()
l1 = 0.0
for param in model.parameters():
  # param in one loop corresponds to all weights of each layer
  print(len(param))
  l1 += torch.sum(param**2)
l1

In [ ]:
loss = loss_function(prediction, true_data) = l1